In [2]:
#Step_1:  Load and Inspect the Data
#Ensure you load the dataset and inspect its structure to understand its contents.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Load the dataset
dataPath = "./Data/TRAIN_Vehicles_Data.xlsx" #The route to the data
data = pd.read_excel(dataPath, na_values = [" ", "N", "Nicht definiert", "Keine Zuteilung","siehe Ausstattung"])
data.drop(columns = 'COLOR')
colors = pd.read_excel("./Data/colors.xlsx")
data['COLOR'] = colors['COLOR']


In [3]:
#Step 2: Visualizing the Target Variable
#Let’s analyze the distribution of LAID_UP_TIME and check for outliers.

#Remove rows where the target (LAID_UP_TIME) is Nan
data = data.dropna(subset=["LAID_UP_TIME"])


In [4]:
#Step 3: Data Cleaning and Preprocessing
nans_thresh = 0.3
nans_percent = data.isna().mean()
valid_columns = [
    col for col in data.columns if (nans_percent[col] <= nans_thresh)
]

data_filtered = data[valid_columns]

In [5]:
#Selected numerical attributes
num_attribs = ['SCALED_INVENTURAL_VALUE', 'AT_LOCATION_SINCE','LEASING_MILAGE',
               'SCALED_REPORT_VALUE','NUMBER_AXLE', 'IS_USED_CAR', 'CURB_WEIGHT', 'PURCHASE_MILAGE', 
               'MILEAGE', 'NUMBER_SEATS', 'YEAR_CONSTRUCTION', 'HORSEPOWER', 'NUMBER_DOORS']

In [6]:
#Number of categories in categorical columns
category_counts = data_filtered['MANUFACTURER_SHORT'].value_counts()
valid_categories = category_counts[category_counts >= 400].index
data_filtered['MANUFACTURER_SHORT'] = data['MANUFACTURER_SHORT'].apply(lambda x: x if x in valid_categories else None)
data_filtered['MANUFACTURER_SHORT'].value_counts()

data_filtered.select_dtypes(include=['object', 'category']).nunique()

/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/1900645259.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_filtered['MANUFACTURER_SHORT'] = data['MANUFACTURER_SHORT'].apply(lambda x: x if x in valid_categories else None)


RPAKREP_VEHICLE_HKEY         87530
OFFICE                         111
OFFICE_MAIN_BRAND               15
CHASSIS_NUMBER               86234
MANUFACTURER_SHORT              23
MANUFACTURER                    97
VEHICLE_GROUP                  613
VEHICLE_TYPE                 13705
MODEL_CODE                    4999
COLOR                           14
UPHOLSTERY                    4534
ENGINE_TYPE                   1224
TRANSMISSION_TYPE              680
TRANSMISSION_SHORT              12
TRANSMISSION_NAME               11
COMMISSION_NUMBER            86591
FINANCING_TYPE_NAME              5
FUEL_TYPE                       13
FUEL_TYPE_NAME                  11
VEHICLE_MODEL_ID_NAME            9
COMMISSION_TYPE                  5
COMMISSION_TYPE_NAME             5
SOLD_CUSTOMER_ID             43941
SOLD_INVOICE_COSTUMER_ID     32681
SOLD_INVOICE_COSTUMER_ID2    32681
SALE_CUSTOMER_ID2            43941
dtype: int64

In [7]:
#Selected categorical attributes
cat_attribs = ['OFFICE_MAIN_BRAND','TRANSMISSION_SHORT', 'FUEL_TYPE',
               'VEHICLE_MODEL_ID_NAME','COMMISSION_TYPE', 'COLOR','MANUFACTURER_SHORT']

In [8]:
targets = data_filtered["LAID_UP_TIME"].copy()
data_cleaned = data_filtered[num_attribs + cat_attribs]
data_cleaned.head()

,SCALED_INVENTURAL_VALUE,AT_LOCATION_SINCE,LEASING_MILAGE,SCALED_REPORT_VALUE,NUMBER_AXLE,IS_USED_CAR,CURB_WEIGHT,PURCHASE_MILAGE,MILEAGE,NUMBER_SEATS,YEAR_CONSTRUCTION,HORSEPOWER,NUMBER_DOORS,OFFICE_MAIN_BRAND,TRANSMISSION_SHORT,FUEL_TYPE,VEHICLE_MODEL_ID_NAME,COMMISSION_TYPE,COLOR,MANUFACTURER_SHORT
0,0.000000,0.0,0.0,0.950043,0.0,1.0,0.0,8600.0,8600.0,5.0,2018.0,140.0,5.0,TOY,Y,1,NaN,2,Grey,FOR
1,0.000000,0.0,20000.0,0.950043,0.0,0.0,1688.0,0.0,0.0,5.0,2025.0,163.0,5.0,VOL,Z,9,Gelaendewagen/Pickup,1,Black,VOL
2,0.000000,0.0,67500.0,0.950043,0.0,0.0,0.0,0.0,1297.0,5.0,NaN,122.0,5.0,SKO,Y,3,Van/Kleinbus,1,Black,FOR
3,0.029537,0.0,0.0,0.897257,0.0,0.0,0.0,0.0,6020.0,0.0,NaN,0.0,0.0,V,NaN,NaN,NaN,1,Silver,V
4,0.000000,1231117.0,30000.0,0.950043,2.0,0.0,0.0,0.0,0.0,0.0,2024.0,125.0,0.0,FOR,NaN,9,Kombi,1,Black,FOR


In [9]:
# Detect columns with mixed types (e.g., strings and integers) and convert to string
def detect_mixed_type_columns(df):
    mixed_columns = []
    for col in df.columns:
        # Get unique data types in the column
        unique_types = df[col].dropna().map(type).unique()
        print(col, unique_types)
        if len(unique_types) > 1:  # More than one type detected
            mixed_columns.append(col)
    return mixed_columns

mixed_columns = detect_mixed_type_columns(data_cleaned)

for col in mixed_columns:
    data_cleaned[col] = data_cleaned[col].astype(str)

SCALED_INVENTURAL_VALUE [<class 'float'>]
AT_LOCATION_SINCE [<class 'float'>]
LEASING_MILAGE [<class 'float'>]
SCALED_REPORT_VALUE [<class 'float'>]
NUMBER_AXLE [<class 'float'>]
IS_USED_CAR [<class 'float'>]
CURB_WEIGHT [<class 'float'>]
PURCHASE_MILAGE [<class 'float'>]
MILEAGE [<class 'float'>]
NUMBER_SEATS [<class 'float'>]
YEAR_CONSTRUCTION [<class 'float'>]
HORSEPOWER [<class 'float'>]
NUMBER_DOORS [<class 'float'>]
OFFICE_MAIN_BRAND [<class 'str'>]
TRANSMISSION_SHORT [<class 'str'> <class 'int'>]
FUEL_TYPE [<class 'int'> <class 'str'>]
VEHICLE_MODEL_ID_NAME [<class 'str'>]
COMMISSION_TYPE [<class 'int'> <class 'str'>]
COLOR [<class 'str'>]
MANUFACTURER_SHORT [<class 'str'>]


/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/2179302096.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned[col] = data_cleaned[col].astype(str)
/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/2179302096.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned[col] = data_cleaned[col].astype(str)
/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/2179302096.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

In [10]:
#Preprocess the data
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('std_scaler', StandardScaler()),
    ])
cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="most_frequent")),
        ("cat", OneHotEncoder())
    ])

num_attribs = data_cleaned.select_dtypes(include=['number']).columns.tolist()
cat_attribs = data_cleaned.select_dtypes(include=['object']).columns.tolist()


full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs)
    ])

data_preprocessed = full_pipeline.fit_transform(data_cleaned)

In [18]:
#Step 4: Model Training and Evaluation
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

X_train, X_test, y_train, y_test = train_test_split(data_preprocessed, targets, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)

param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10),
}

# Set up the RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, 
                                   n_iter=5, cv=1, scoring='neg_mean_squared_error', 
                                   verbose=2, random_state=42, n_jobs=-1)



model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.4f}")

RMSE: 79.9471


In [19]:
#Loading test data
dataPath = "./Data/Vehicles_export_prices_scaled_stud_test_eng-2.xlsx" #The route to the data
test = pd.read_excel(dataPath, na_values = [" ", "N", "Nicht definiert", "Keine Zuteilung"])
test.drop(columns = 'COLOR')
colors = pd.read_excel("./Data/colortest2.xlsx")
test['COLOR'] = colors['COLOR']


category_counts = test['MANUFACTURER_SHORT'].value_counts()
valid_categories = category_counts[category_counts >= 400].index
test['MANUFACTURER_SHORT'] = test['MANUFACTURER_SHORT'].apply(lambda x: x if x in valid_categories else None)
test['MANUFACTURER_SHORT'].value_counts()

#Same preprocessing for test data
test_cleaned = test[num_attribs + cat_attribs]

mixed_columns = detect_mixed_type_columns(test_cleaned)

for col in mixed_columns:
    test_cleaned[col] = test_cleaned[col].astype(str)

test_preprocessed = full_pipeline.transform(test_cleaned)

#Train and predict
model.fit(data_preprocessed, targets)
predictions = model.predict(test_preprocessed)
results = pd.DataFrame({
    'CHASSIS_NUMBER': test['CHASSIS_NUMBER'],
    'LAID_UP_TIME': predictions
})

results_path = "Results_2.xlsx"
results.to_excel(results_path, index=False)

SCALED_INVENTURAL_VALUE [<class 'float'>]
AT_LOCATION_SINCE [<class 'int'>]
LEASING_MILAGE [<class 'int'>]
SCALED_REPORT_VALUE [<class 'float'>]
NUMBER_AXLE [<class 'int'>]
IS_USED_CAR [<class 'int'>]
CURB_WEIGHT [<class 'int'>]
PURCHASE_MILAGE [<class 'int'>]
MILEAGE [<class 'int'>]
NUMBER_SEATS [<class 'int'>]
YEAR_CONSTRUCTION [<class 'float'>]
HORSEPOWER [<class 'int'>]
NUMBER_DOORS [<class 'int'>]
OFFICE_MAIN_BRAND [<class 'str'>]
TRANSMISSION_SHORT [<class 'str'> <class 'int'>]
FUEL_TYPE [<class 'int'> <class 'str'>]
VEHICLE_MODEL_ID_NAME [<class 'str'>]
COMMISSION_TYPE [<class 'int'> <class 'str'>]
COLOR [<class 'str'>]
MANUFACTURER_SHORT [<class 'str'>]


/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/2328291186.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_cleaned[col] = test_cleaned[col].astype(str)
/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/2328291186.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_cleaned[col] = test_cleaned[col].astype(str)
/var/folders/v3/tl2b04r50k3gjhc2srfr8sv00000gn/T/ipykernel_33798/2328291186.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli